# Funding-carry backtest — net of costs

Does the carry survive fees and borrow? Simulates a **delta-neutral** hold over the 30d funding window (`hl_funding_hist`): collect funding on the side that pays (short perp + long spot when funding>0; long perp + short spot when <0), then subtract realistic costs. Returns are **per unit notional, unlevered (1×)** — leverage scales both the return and the risk.

**This is in-sample on 30 days** — a forward-looking number needs the carry to persist; see the spike analysis in `12_funding_carry`.

In [ ]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
f = con.execute('SELECT coin, t, funding_rate FROM hl_funding_hist ORDER BY t').df()
con.close()
# --- cost assumptions (edit these) ---
FEE_PERP, FEE_SPOT = 0.00035, 0.0010   # taker fee per fill
BORROW_APR = 0.25                       # annual cost to borrow spot for the SHORT-spot leg
HOURS_Y = 24*365
piv = f.pivot_table(index='t', columns='coin', values='funding_rate').sort_index()
n_h = len(piv); window_y = n_h/HOURS_Y
print(f'{n_h} hourly steps ({window_y*365:.0f} days), {piv.shape[1]} coins')

## Per-coin: gross vs net carry

`gross` = funding collected on the paying side. `fee` = entry+exit on both legs, amortized over the hold. `borrow` = only when the carry requires shorting spot (negative funding).

In [ ]:
rows=[]
total_fee = 2*(FEE_PERP+FEE_SPOT)              # open+close, both legs
fee_apr = total_fee/window_y
for coin in piv.columns:
    s = piv[coin].dropna()
    if len(s) < 100: continue
    direction = np.sign(s.mean())              # +1 short-perp/long-spot ; -1 long-perp/short-spot
    shorts_spot = direction < 0
    gross_apr = (s*direction).mean()*HOURS_Y*100
    borrow_apr = (BORROW_APR*100) if shorts_spot else 0.0
    net_apr = gross_apr - fee_apr*100 - borrow_apr
    rows.append({'coin':coin,'side':'short-perp' if direction>0 else 'long-perp',
                 'gross_%':round(gross_apr,1),'fee_%':round(fee_apr*100,1),
                 'borrow_%':round(borrow_apr,1),'net_%':round(net_apr,1),
                 'stability':round(float((np.sign(s)==direction).mean()),2)})
res = pd.DataFrame(rows).sort_values('net_%', ascending=False).reset_index(drop=True)
res

## Gross vs net (top carries)

The gap is the cost drag; bars that stay tall after costs are the real ones.

In [ ]:
d = res.head(14).iloc[::-1]
y=np.arange(len(d)); plt.figure(figsize=(10,6))
plt.barh(y+0.2, d['gross_%'], height=0.4, color='steelblue', label='gross')
plt.barh(y-0.2, d['net_%'], height=0.4, color='seagreen', label='net of costs')
plt.axvline(0,color='crimson',lw=.6); plt.yticks(y, d['coin']); plt.xlabel('annualized %'); plt.legend()
plt.title('Funding carry: gross vs net'); plt.tight_layout(); plt.show()

## Portfolio backtest — equity curve

Equal-weight the coins with **net carry > 5% and sign-stability > 0.7** (the durable ones), hold 30d, accrue funding hourly minus amortized costs. Unlevered.

In [ ]:
basket = res[(res['net_%']>5) & (res['stability']>0.7)].coin.tolist()
print(f'basket ({len(basket)}):', basket)
if basket:
    hourly_fee = total_fee/n_h
    legs=[]
    for coin in basket:
        s = piv[coin]; direction=np.sign(s.mean()); shorts=direction<0
        net_h = s*direction - hourly_fee - (BORROW_APR/HOURS_Y if shorts else 0)
        legs.append(net_h.rename(coin))
    port = pd.concat(legs, axis=1).mean(axis=1).dropna()
    eq = (1+port).cumprod()
    ann = port.mean()*HOURS_Y*100; vol = port.std()*np.sqrt(HOURS_Y)*100
    sharpe = ann/vol if vol else float('nan'); dd = ((eq/eq.cummax())-1).min()*100
    fig,ax=plt.subplots(figsize=(11,5)); ax.plot(eq.index, (eq-1)*100, color='seagreen')
    ax.set_ylabel('cumulative net return %'); ax.set_title('Delta-neutral funding-carry basket (unlevered, net of costs)')
    ax.axhline(0,color='gray',lw=.6); plt.show()
    print(f'net APR {ann:.1f}% | vol {vol:.1f}% | Sharpe {sharpe:.2f} | max drawdown {dd:.2f}% | {len(basket)} coins')
else:
    print('no coins clear the net>5% / stability>0.7 bar after costs.')

## Verdict

**Read the net APR, ignore the Sharpe.** The reported vol / Sharpe / max-drawdown are **misleading artifacts** — they only measure how *smooth the funding trickle* is (funding barely varies hour to hour, so vol ≈ 0 and Sharpe explodes to ~88). They do **not** include the actual risks of the position: perp-vs-spot **basis divergence**, **liquidation** on the perp leg, and **funding regime-shift**. A real carry book has meaningful drawdowns from those; this model shows ~0 because it doesn't simulate them. So the Sharpe/DD here are not trustworthy — only the **net carry rate** is.

On the rates: **positive-funding carries** (short perp + *hold* spot) only pay trading fees, so a stable name like **XMR keeps ~37.6% net** of its ~41% gross — the standout. The liquid majors (HYPE/ETH/DOGE) net a *modest ~4–5%*. **Negative-funding carries** (CHIP 68%, STABLE 30%) look huge but require **borrowing spot**, and that 25% `BORROW_APR` is optimistic for thin alts (often higher or unavailable) — treat those nets as a ceiling. The 4-coin basket's 32% is inflated by exactly those optimistic alts (CHIP) and a spike (VVV, per `12_funding_carry`); the *robust* core is XMR + the small liquid carries.

Also not captured: **in-sample over 30d** (funding can regime-shift), **capacity** (fat carries are thin — can't size up), slippage, and that funding accrues on *notional* while margin is a fraction (leverage scales returns AND liquidation risk). Honest bottom line: funding carry is a **real but modest, operationally fiddly yield** — XMR-like stable positive-funding names net the most cleanly; everything else is smaller or riskier than the headline. It's the closest thing to an edge in this survey, and it is *not* free money.